In [1]:

import pandas as pd
import numpy as np
from pathlib import Path

path = Path(
    "results/matched_15eval_seed42/"
    "gaussian_process_random_forest_random_forest_expanded_seed42/"
    "candidates.csv"
)

df = pd.read_csv(path)

print("\n==============================")
print("CSV COLUMNS")
print("==============================")
print(df.columns.tolist())

print("\n==============================")
print("ALL CANDIDATES")
print("==============================")
print(df.to_string(index=False))

# Remove the 5 random warm-start points.
opt = df[df["surrogate_source"].isin(
    ["gaussian_process", "random_forest"]
)].copy()

# Identify expected columns.
n_estimators_col = next(
    (c for c in df.columns if "n_estimators" in c),
    None
)
max_depth_col = next(
    (c for c in df.columns if "max_depth" in c),
    None
)

duration_col = next(
    (
        c for c in [
            "eval_duration",
            "evaluation_duration",
            "eval_time",
            "evaluation_time",
        ]
        if c in df.columns
    ),
    None
)

# If duration isn't explicitly stored, calculate it.
if duration_col is None:
    if {"eval_start_time", "eval_end_time"}.issubset(df.columns):
        opt["computed_eval_duration"] = (
            opt["eval_end_time"] - opt["eval_start_time"]
        )
        duration_col = "computed_eval_duration"

print("\n==============================")
print("DETECTED FIELDS")
print("==============================")
print("n_estimators:", n_estimators_col)
print("max_depth:", max_depth_col)
print("evaluation duration:", duration_col)

required = [n_estimators_col, max_depth_col, duration_col]

if any(c is None for c in required):
    print(
        "\nCould not automatically detect all required fields.\n"
        "Use the column list printed above and inspect the schema."
    )
    raise SystemExit(1)

cols = [
    "surrogate_source",
    n_estimators_col,
    max_depth_col,
    "y",
    duration_col,
]

if "round_id" in opt.columns:
    cols.insert(0, "round_id")

print("\n==============================")
print("GP VS RF CANDIDATES")
print("==============================")
print(
    opt[cols]
    .to_string(index=False)
)

print("\n==============================")
print("SUMMARY BY SURROGATE")
print("==============================")

summary = opt.groupby("surrogate_source").agg(
    n_candidates=("y", "size"),

    mean_n_estimators=(n_estimators_col, "mean"),
    median_n_estimators=(n_estimators_col, "median"),
    min_n_estimators=(n_estimators_col, "min"),
    max_n_estimators=(n_estimators_col, "max"),

    mean_max_depth=(max_depth_col, "mean"),
    median_max_depth=(max_depth_col, "median"),

    mean_eval_time=(duration_col, "mean"),
    median_eval_time=(duration_col, "median"),
    min_eval_time=(duration_col, "min"),
    max_eval_time=(duration_col, "max"),

    mean_y=("y", "mean"),
    median_y=("y", "median"),
    best_y=("y", "max"),
)

print(summary.round(4).to_string())

print("\n==============================")
print("CORRELATIONS — ALL GP/RF CANDIDATES")
print("==============================")

corr_cols = [
    n_estimators_col,
    max_depth_col,
    duration_col,
    "y",
]

print(
    opt[corr_cols]
    .corr(numeric_only=True)
    .round(3)
    .to_string()
)

print("\n==============================")
print("SIMPLE INTERPRETATION HELP")
print("==============================")

for source in ["gaussian_process", "random_forest"]:
    rows = opt[opt["surrogate_source"] == source]

    print(f"\n{source}:")
    print(
        f"  n_estimators range: "
        f"{rows[n_estimators_col].min()} - "
        f"{rows[n_estimators_col].max()}"
    )
    print(
        f"  mean n_estimators: "
        f"{rows[n_estimators_col].mean():.1f}"
    )
    print(
        f"  mean evaluation time: "
        f"{rows[duration_col].mean():.2f}s"
    )
    print(
        f"  mean objective: "
        f"{rows['y'].mean():.6f}"
    )
    print(
        f"  best objective: "
        f"{rows['y'].max():.6f}"
    )


FileNotFoundError: [Errno 2] No such file or directory: 'results/matched_15eval_seed42/gaussian_process_random_forest_random_forest_expanded_seed42/candidates.csv'